In [1]:
import os
import ast
import random
import pandas as pd
import numpy as np
import implicit
import joblib

from scipy.sparse import csr_matrix

c:\Users\ghgd4\Desktop\MAI\2.2\diplom\code\global_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df_movies = pd.read_csv("data/movies_train.csv")
print(f"Всего фильмов в базе: {len(df_movies)}")

df_movies['genres_list'] = df_movies['genres_list'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

all_genres = []
for genres in df_movies['genres_list']:
    all_genres.extend(genres)
all_genres = list(set(all_genres))
print(f"Всего жанров: {len(all_genres)}")

ratings = []
n_users = 2000
min_ratings_per_user = 20
max_ratings_per_user = 60

random.seed(42)
np.random.seed(42)

for user_id in range(1, n_users + 1):
    k = min(random.randint(2, 4), len(all_genres))
    fav_genres = random.sample(all_genres, k=k)

    candidate_movies = df_movies[
        df_movies['genres_list'].apply(lambda x: any(g in x for g in fav_genres))
    ]

    if len(candidate_movies) == 0:
        continue

    n_ratings = random.randint(min_ratings_per_user, min(max_ratings_per_user, len(candidate_movies)))

    raw = candidate_movies['votes_kp'].fillna(0).values.astype(float)
    weights = np.log1p(raw)

    if weights.sum() == 0:
        weights = None
    else:
        weights = weights / weights.sum()
        max_allowed = 1.0 / n_ratings
        guard = 0
        while weights.max() > max_allowed and guard < 20:
            weights = np.power(weights, 0.5)
            weights = weights / weights.sum()
            guard += 1
        if weights.max() > max_allowed:
            weights = None

    selected = candidate_movies.sample(
        n=n_ratings, random_state=user_id, weights=weights
    )

    for _, movie in selected.iterrows():
        base = movie['rating_kp']
        match_ratio = len(set(fav_genres) & set(movie['genres_list'])) / len(fav_genres)
        genre_bonus = (match_ratio - 0.5) * 4
        noise = np.random.normal(0, 1)
        rating = base + genre_bonus + noise
        rating = int(np.clip(round(rating), 1, 10))

        ratings.append({
            'user_id': user_id,
            'movie_id': movie['id'],
            'rating': rating,
            'title': movie['name']
        })

print()
df_ratings = pd.DataFrame(ratings)
print(f"Создано {df_ratings['user_id'].nunique()} пользователей")
print(f"Всего оценок: {len(df_ratings)}")
print(f"Уникальных оценённых фильмов: {df_ratings['movie_id'].nunique()} из {len(df_movies)}")
print(f"Средняя оценка: {df_ratings['rating'].mean():.2f}")
print(f"Средний rating_kp: {df_movies['rating_kp'].mean():.2f}")

df_ratings.to_csv("data/user_ratings_emulated.csv", index=False)
print("Сохранено: data/user_ratings_emulated.csv")

print()
display(df_ratings.head(10))

Фильмов в базе: 17017
Всего жанров: 23

Создано 2000 пользователей
Всего оценок: 80577
Уникальных оценённых фильмов: 15879 из 17017
Средняя оценка: 6.14
Средний rating_kp: 6.50
Сохранено: data/user_ratings_emulated.csv



,user_id,movie_id,rating,title
0,1,581932,6,Перси Джексон и Море чудовищ
1,1,446136,5,Tomb Raider: Лара Крофт
2,1,448,9,Форрест Гамп
3,1,450760,9,Знаки
4,1,613,6,Блэйд 2
5,1,7574,6,Безумие короля Георга
6,1,77155,6,От заката до рассвета 3: Дочь палача
7,1,373110,7,Ло
8,1,566283,5,Сквозь снег
9,1,484628,6,Тайна 7 сестер


In [ ]:
users = df_ratings['user_id'].unique()
np.random.seed(42)
test_users = set(np.random.choice(users, size=int(len(users) * 0.2), replace=False))

train_data, test_data = [], []
for user_id, user_ratings in df_ratings.groupby('user_id'):
    if len(user_ratings) < 5:
        continue
    if user_id in test_users:
        hidden = user_ratings.sample(n=min(3, len(user_ratings) - 1), random_state=42)
        train_data.append(user_ratings.drop(hidden.index))
        test_data.append(hidden)
    else:
        train_data.append(user_ratings)

train_df = pd.concat(train_data)
test_df = pd.concat(test_data)
print(f"train: {len(train_df)}  test: {len(test_df)}  тестовых юзеров: {len(test_users)}")

user_ids = df_ratings['user_id'].unique()
movie_ids = df_ratings['movie_id'].unique()
user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

row = train_df['user_id'].map(user_to_idx).values
col = train_df['movie_id'].map(movie_to_idx).values
data = train_df['rating'].values.astype(float)

matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), len(movie_ids)))
model = implicit.als.AlternatingLeastSquares(factors=50, iterations=30, random_state=42)
model.fit(matrix)

os.makedirs('models/current', exist_ok=True)
joblib.dump(model, 'models/current/als_model.pkl')
joblib.dump(user_to_idx, 'models/current/user_to_idx.pkl')
joblib.dump(movie_to_idx, 'models/current/movie_to_idx.pkl')
joblib.dump(user_ids, 'models/current/user_ids.pkl')
joblib.dump(movie_ids, 'models/current/movie_ids.pkl')

train_df.to_csv('data/train_ratings.csv', index=False)
test_df.to_csv('data/test_ratings.csv', index=False)

print("Сохранено в models/current/ и data/")

c:\Users\ghgd4\Desktop\MAI\2.2\diplom\code\global_venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 30/30 [00:00<00:00, 46.24it/s]


Сохранено в models/current/
